# Statistik mit Python: vom Rohdatensatz zur Aussage

**Einstieg für Statistik in der Informatik / Wirtschaftsinformatik**

Heute geht es nicht darum, Python „zu lernen“. Wir benutzen Python als **Werkzeug**, um statistische Fragen an Daten zu stellen.

## Lernziele

Nach diesem Notebook können Sie …

- einen Datensatz in einem Jupyter Notebook laden und inspizieren,
- Zeilen und Spalten auswählen und Daten filtern,
- Häufigkeiten sowie Lage- und Streuungsmaße berechnen,
- Histogramm, Balkendiagramm, Boxplot und Streudiagramm sinnvoll einsetzen,
- Gruppen vergleichen,
- Korrelation vorsichtig interpretieren,
- erklären, warum Stichproben zufällig schwanken.

> **Bedienung:** Eine Zelle mit `Shift` + `Enter` ausführen.  
> Code darf verändert werden – genau dafür ist ein Notebook da.

---

### Die Leitfrage

Wir betrachten Messdaten eines fiktiven Web-Systems. Jede Zeile ist ein API-Request.

**Fragen für heute:**

1. Welche Endpoints werden häufig aufgerufen?
2. Welcher Endpoint ist typischerweise langsam?
3. Wie stark streuen Antwortzeiten?
4. Haben große Payloads längere Antwortzeiten?
5. Macht ein Cache-Hit einen Unterschied?
6. Wie stabil ist ein Mittelwert, wenn wir nur eine Stichprobe beobachten?

## 0. R und Python – unterschiedliche Syntax, gleiche Statistik

Die verlinkte Statistik-Veranstaltung arbeitet überwiegend mit **R**. Hier verwenden wir **Python**.  
Die statistischen Konzepte sind dieselben.

| Aufgabe | R | Python / pandas |
|---|---|---|
| Mittelwert | `mean(x)` | `x.mean()` |
| Median | `median(x)` | `x.median()` |
| Standardabweichung | `sd(x)` | `x.std()` |
| Häufigkeiten | `table(x)` | `x.value_counts()` |
| Zusammenfassung | `summary(x)` | `x.describe()` |
| Histogramm | `hist(x)` | `x.plot.hist()` |
| Gruppieren | `aggregate(...)` / `dplyr` | `groupby(...)` |

**Merksatz:** Statistik ist die Idee – Software ist das Werkzeug.

## 1. Werkzeugkasten laden

Für heute reichen drei verbreitete Pakete:

- `pandas` für Tabellen und Datenmanipulation,
- `numpy` für numerische Operationen und Zufall,
- `matplotlib` für Diagramme.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (8, 4.5)

## 2. Datensatz laden

Der Datensatz liegt als CSV-Datei im Unterordner `data/`.

**Statistische Einheit:** ein API-Request.

Einige Merkmale:

- `endpoint`, `region`: kategorial
- `hour`, `payload_kb`, `response_ms`: quantitativ
- `cache_hit`: binär
- `status_code`: numerisch codiert, inhaltlich aber eher kategorial

In [ ]:
df = pd.read_csv("data/api_requests.csv")
df.head()

In [ ]:
print("Zeilen, Spalten:", df.shape)
print("\nSpalten:")
print(df.columns.to_list())

### Erster Blick auf die Daten

Bevor wir rechnen, prüfen wir Struktur, Datentypen und fehlende Werte.

In [ ]:
df.info()

In [ ]:
df.isna().sum()

Wir haben einige fehlende Antwortzeiten. Für die folgenden Analysen arbeiten wir mit einer bereinigten Kopie.

In [ ]:
clean = df.dropna(subset=["response_ms"]).copy()
print("Vorher:", len(df), "Zeilen")
print("Nachher:", len(clean), "Zeilen")

## 3. Daten auswählen und filtern

Das ist für Informatiker:innen oft der intuitivste Einstieg:  
Ein DataFrame ist eine Tabelle, und Bedingungen liefern Wahrheitswerte (`True`/`False`).

In [ ]:
# Eine Spalte auswählen
clean["response_ms"].head()

In [ ]:
# Alle Serverfehler (HTTP 500)
server_errors = clean[clean["status_code"] == 500]
server_errors[["request_id", "endpoint", "response_ms", "status_code"]].head()

In [ ]:
# Mehrere Bedingungen mit & verknüpfen
slow_checkout = clean[
    (clean["endpoint"] == "/checkout")
    & (clean["response_ms"] > 400)
]

slow_checkout.head()

### 🧪 Mini-Aufgabe 1

Filtern Sie alle Requests, die

- den Endpoint `/search` verwenden,
- aus `EU-Central` kommen,
- und länger als **250 ms** dauern.

Wie viele sind es?

In [ ]:
# TODO: Schreiben Sie hier Ihren Filter.
# Tipp: Verwenden Sie drei Bedingungen und len(...).

## 4. Kategoriale Daten: Häufigkeiten

Bei kategorialen Merkmalen ist eine der ersten Fragen:

> **Wie oft kommt jede Ausprägung vor?**

In [ ]:
clean["endpoint"].value_counts()

In [ ]:
# Relative Häufigkeiten
(clean["endpoint"].value_counts(normalize=True) * 100).round(1)

In [ ]:
clean["endpoint"].value_counts().plot(kind="bar")
plt.title("Anzahl Requests je Endpoint")
plt.xlabel("Endpoint")
plt.ylabel("Anzahl")
plt.xticks(rotation=0)
plt.show()

### Welche Darstellung passt?

- **Balkendiagramm:** Kategorien und ihre Häufigkeiten
- **Histogramm:** Verteilung einer metrischen Größe
- **Boxplot:** kompakter Vergleich von Verteilungen
- **Streudiagramm:** Zusammenhang zweier metrischer Größen

## 5. Metrische Daten: Lage und Streuung

Wir betrachten die Antwortzeit `response_ms`.

In [ ]:
clean["response_ms"].describe()

In [ ]:
x = clean["response_ms"]

print("Mittelwert:          ", round(x.mean(), 1), "ms")
print("Median:              ", round(x.median(), 1), "ms")
print("Standardabweichung:  ", round(x.std(), 1), "ms")
print("25%-Quantil:         ", round(x.quantile(0.25), 1), "ms")
print("75%-Quantil:         ", round(x.quantile(0.75), 1), "ms")

### Mittelwert oder Median?

Der Mittelwert reagiert empfindlicher auf extreme Werte.  
Schauen wir uns die langsamsten Requests an.

In [ ]:
clean.nlargest(8, "response_ms")[
    ["request_id", "endpoint", "region", "response_ms", "status_code"]
]

In [ ]:
clean["response_ms"].plot.hist(bins=28, edgecolor="black")
plt.axvline(clean["response_ms"].mean(), linestyle="--", label="Mittelwert")
plt.axvline(clean["response_ms"].median(), linestyle=":", label="Median")
plt.title("Verteilung der Antwortzeiten")
plt.xlabel("Antwortzeit [ms]")
plt.ylabel("Häufigkeit")
plt.legend()
plt.show()

In [ ]:
clean[["response_ms"]].boxplot()
plt.title("Boxplot der Antwortzeiten")
plt.ylabel("Antwortzeit [ms]")
plt.show()

**Interpretation:** Bei rechtsschiefen Daten mit einzelnen sehr langsamen Requests liegt der Mittelwert typischerweise über dem Median.

Der Median beantwortet eher: *„Wie lange dauert ein typischer Request?“*  
Der Mittelwert beantwortet eher: *„Wie groß ist die durchschnittliche Last/Verzögerung über alle Requests?“*

## 6. Gruppen vergleichen: `groupby`

Jetzt wird die Analyse interessanter:

> Unterscheiden sich die Antwortzeiten zwischen den Endpoints?

In [ ]:
endpoint_stats = (
    clean.groupby("endpoint")["response_ms"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("median")
    .round(1)
)

endpoint_stats

In [ ]:
clean.boxplot(column="response_ms", by="endpoint", grid=False)
plt.suptitle("")
plt.title("Antwortzeiten nach Endpoint")
plt.xlabel("Endpoint")
plt.ylabel("Antwortzeit [ms]")
plt.xticks(rotation=20)
plt.show()

### 🧪 Mini-Aufgabe 2

Vergleichen Sie Requests **mit** und **ohne** Cache-Hit.

Berechnen Sie für beide Gruppen:

- Anzahl,
- Mittelwert,
- Median der Antwortzeit.

**Frage:** Reicht dieser Vergleich schon aus, um zu behaupten, dass Caching *kausal* eine bestimmte Anzahl Millisekunden spart?

In [ ]:
# TODO: Gruppieren Sie nach "cache_hit" und analysieren Sie "response_ms".

## 7. Zwei metrische Merkmale: Streudiagramm und Korrelation

Frage:

> Sind größere Payloads mit längeren Antwortzeiten verbunden?

In [ ]:
clean.plot.scatter(x="payload_kb", y="response_ms", alpha=0.6)
plt.title("Payload-Größe und Antwortzeit")
plt.xlabel("Payload [kB]")
plt.ylabel("Antwortzeit [ms]")
plt.show()

In [ ]:
corr = clean["payload_kb"].corr(clean["response_ms"])
print("Pearson-Korrelation:", round(corr, 3))

Eine Korrelation nahe `+1` bedeutet einen starken positiven linearen Zusammenhang, nahe `-1` einen starken negativen linearen Zusammenhang und nahe `0` keinen starken **linearen** Zusammenhang.

> **Wichtig:** Korrelation ist kein Beweis für Kausalität.

In [ ]:
# Eine einfache lineare Trendlinie – nur als visuelle Orientierung
x = clean["payload_kb"].to_numpy()
y = clean["response_ms"].to_numpy()

m, b = np.polyfit(x, y, 1)

plt.scatter(x, y, alpha=0.55)
x_line = np.linspace(x.min(), x.max(), 100)
plt.plot(x_line, m * x_line + b)
plt.title("Payload-Größe und Antwortzeit mit linearer Trendlinie")
plt.xlabel("Payload [kB]")
plt.ylabel("Antwortzeit [ms]")
plt.show()

print(f"Trendlinie: response_ms ≈ {m:.2f} · payload_kb + {b:.1f}")

## 8. Warum Stichproben schwanken

Statistik beschäftigt sich oft nicht mit **allen** Daten, sondern mit einer Stichprobe.

Wir ziehen wiederholt nur 30 Requests und berechnen jedes Mal den Mittelwert.

In [ ]:
rng = np.random.default_rng(42)

sample_means = []
for _ in range(500):
    sample = clean["response_ms"].sample(n=30, replace=False, random_state=int(rng.integers(1_000_000)))
    sample_means.append(sample.mean())

sample_means = pd.Series(sample_means)
sample_means.describe()

In [ ]:
sample_means.plot.hist(bins=25, edgecolor="black")
plt.axvline(clean["response_ms"].mean(), linestyle="--", label="Mittelwert aller beobachteten Requests")
plt.title("500 Stichprobenmittelwerte (je n = 30)")
plt.xlabel("Stichprobenmittelwert der Antwortzeit [ms]")
plt.ylabel("Häufigkeit")
plt.legend()
plt.show()

**Beobachtung:** Obwohl alle Stichproben aus denselben Daten stammen, liefern sie nicht exakt denselben Mittelwert.

Genau hier kommt Wahrscheinlichkeitstheorie ins Spiel:  
Wir wollen verstehen, **wie stark statistische Kennzahlen zufällig schwanken** und welche Schlussfolgerungen aus einer Stichprobe gerechtfertigt sind.

## 9. Abschluss-Challenge

Beantworten Sie mit Python drei Fragen:

1. Welcher Endpoint hat den höchsten **Median** der Antwortzeit?
2. Wie hoch ist der Anteil der Requests mit Statuscode `500` in Prozent?
3. Welche Region hat den höchsten Median der Antwortzeit?

Formulieren Sie zu jeder Zahl einen vollständigen Satz – nicht nur Code.

In [ ]:
# TODO 1: Endpoint mit höchstem Median

In [ ]:
# TODO 2: Anteil HTTP 500 in Prozent

In [ ]:
# TODO 3: Region mit höchstem Median

## 10. Take-away

Heute haben wir denselben grundlegenden Analyseprozess mehrfach benutzt:

**Frage → Daten auswählen → Kennzahl/Diagramm → Interpretation**

Python hat uns dabei Arbeit abgenommen. Die statistische Entscheidung bleibt aber bei uns:

- Welche Beobachtungseinheit betrachten wir?
- Welche Art von Merkmal liegt vor?
- Welche Kennzahl beantwortet unsere Frage?
- Welches Diagramm passt?
- Was dürfen wir aus dem Ergebnis wirklich schließen?

### Zum Weiterprobieren

Ändern Sie Filter, Gruppierungen oder Diagramme. Gute nächste Fragen wären zum Beispiel:

- Gibt es eine „Rush Hour“?
- Sind Fehler-Requests langsamer?
- Unterscheidet sich der Cache-Effekt zwischen `/search` und `/product`?
- Was passiert mit Mittelwert und Median, wenn Sie die fünf langsamsten Requests entfernen?